# Figure 4 notebook

This notebook consolidates the figure-building steps that were previously spread across:

- `05d_view_regions.ipynb`
- `06a_Viz_region_opt.ipynb`
- `06b_joint_percolation.ipynb`

The goal is to reproduce the individual Figure 4 panels and assemble a clean composite mock that matches the attached reference layout.

## What this notebook does

1. Loads the `TissueMultiGraph` object and the precomputed percolation outputs used for Figure 4.
2. Recreates panels `a` through `k`.
3. Saves each panel into `Figures/` as `Figure4_panel_*.png`.
4. Assembles a final `Figure4_mock.png` preview with the same panel organization as the reference figure.

This notebook assumes the project is run from either the repository root or the `Notebooks/` directory.

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import colorcet as cc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import entropy

try:
    from statsmodels.nonparametric.smoothers_lowess import lowess
except ModuleNotFoundError:
    lowess = None

ROOT = Path.cwd().resolve()
if not (ROOT / "Data").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "Data"
FIG_DIR = ROOT / "Figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

for repo_path in [ROOT / "Repos" / "TMG", ROOT / "Repos" / "max_info_atlas" / "src"]:
    repo_str = str(repo_path)
    if repo_str not in sys.path:
        sys.path.insert(0, repo_str)

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.grid": False,
        "savefig.facecolor": "white",
        "font.size": 10,
    }
)

warnings.filterwarnings("ignore", category=UserWarning)

from max_info_atlases.percolation import GraphPercolation
from TMG.Analysis.TissueGraph import Taxonomy, TissueMultiGraph
from TMG.Utils import tmgu
from TMG.Utils.coloru import rgb_array_to_hex
from TMG.Visualization import Viz

CCF_LEVELS = [
    "parcellation_organ",
    "parcellation_category",
    "parcellation_division",
    "parcellation_structure",
    "parcellation_substructure",
]

SECTION_PANEL_SPECS = [
    ("ccf_structure", "c", "d"),
    ("ccf_topdown", "e", "f"),
    ("opt_region", "g", "h"),
]

PANEL_PATHS = {label: FIG_DIR / f"Figure4_panel_{label}.png" for label in "abcdefghijk"}
PANEL_PATHS["mock"] = FIG_DIR / "Figure4_mock.png"


def save_png(fig, path):
    fig.savefig(path, dpi=300, bbox_inches="tight")
    return path


def show_image(ax, image_path):
    image = plt.imread(image_path)
    ax.imshow(image)
    ax.set_axis_off()


def add_panel_label(ax, label):
    ax.text(
        -0.06,
        1.03,
        label,
        transform=ax.transAxes,
        fontsize=16,
        fontweight="bold",
        va="top",
        ha="right",
    )


def smooth_curve(x, y, frac=0.2):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if x.size == 0:
        return x, y
    if lowess is not None and x.size >= 5:
        smoothed = lowess(y, x, frac=frac, return_sorted=True)
        return smoothed[:, 0], smoothed[:, 1]

    window = max(3, int(np.ceil(frac * x.size)))
    if window % 2 == 0:
        window += 1
    if x.size < window:
        return x, y

    kernel = np.ones(window, dtype=float) / window
    y_pad = np.pad(y, (window // 2, window // 2), mode="edge")
    y_smooth = np.convolve(y_pad, kernel, mode="valid")
    return x, y_smooth


def build_ccf_type_matrix(tmg):
    ccf_taxs = {}
    ccf_type_mat = np.zeros((tmg.N[0], len(CCF_LEVELS)), dtype=int)
    tax_basepath = str(DATA_DIR / "TMG2" / "Taxonomies")

    for idx, level in enumerate(CCF_LEVELS):
        values = np.asarray(tmg.Layers[0].adata.obs[level])
        ccf_taxs[level] = Taxonomy(name=level, basepath=tax_basepath, Types=np.unique(values))
        ccf_type_mat[:, idx] = np.asarray(ccf_taxs[level].get_type_ix(values), dtype=int)

    return ccf_taxs, ccf_type_mat


def load_gp_matrix(sections, levels, base_dir):
    gp_matrix = np.empty((len(sections), len(levels)), dtype=object)
    for col, level in enumerate(levels):
        for row, section in enumerate(sections):
            gp = GraphPercolation(np.zeros(1), np.zeros(1))
            gp.load(str(base_dir / level / f"{section}.npz"))
            gp_matrix[row, col] = gp
    return gp_matrix


def load_gp_array_from_dir(sections, base_dir, suffix):
    gp_array = np.empty(len(sections), dtype=object)
    for row, section in enumerate(sections):
        gp = GraphPercolation(np.zeros(1), np.zeros(1))
        gp.load(str(base_dir / f"{section}{suffix}"))
        gp_array[row] = gp
    return gp_array


def load_merge_cache_or_compute(cache_path, gp_matrix, top_down):
    if top_down:
        keys = ("type_ix_td", "best_score_td", "type_vec_td")
    else:
        keys = ("type_ix_bu", "best_score_bu", "type_vec_bu")

    if cache_path.exists():
        cache = np.load(cache_path, allow_pickle=True)
        return cache[keys[0]], float(cache[keys[1]]), cache[keys[2]]

    type_ix, best_score, type_vec = tmgu.merge_nested_clusters(gp_matrix, top_down=top_down)
    np.savez(cache_path, **{keys[0]: type_ix, keys[1]: best_score, keys[2]: type_vec})
    return type_ix, float(best_score), type_vec


def make_section_map(tmg, section, tax_name, panel_key):
    view = Viz.SingleMapView(
        tmg,
        section=section,
        level_type=tax_name,
        rotation=180,
        figsize=(4, 2),
    )
    view.show()
    view.Panels[0].ax.set_title("")
    save_png(view.fig, PANEL_PATHS[panel_key])
    plt.show()
    return view


def make_stackplot(tmg, tax_name, panel_key):
    tmg.update_current_type(0, tax_name)
    sec_reg_xtab = pd.crosstab(tmg.Layers[0].Type, tmg.Layers[0].Section)
    try:
        colors = rgb_array_to_hex(tmg.get_tax(tax_name).RGB)[: len(sec_reg_xtab.index)]
    except Exception:
        colors = cc.glasbey_bw[: len(sec_reg_xtab.index)]

    fig, ax = plt.subplots(figsize=(4, 2))
    x = np.arange(sec_reg_xtab.shape[1])
    y_stack = np.vstack([sec_reg_xtab.iloc[i].values for i in range(sec_reg_xtab.shape[0])])
    ax.stackplot(x, y_stack, colors=colors, alpha=1.0)
    ax.set_xlabel("Anterior-Posterior", fontsize=7)
    ax.set_ylabel("Cell Count", fontsize=7)
    ax.set_xticks([])
    ax.tick_params(axis="y", labelsize=6)
    plt.tight_layout()
    save_png(fig, PANEL_PATHS[panel_key])
    plt.show()
    return fig


tab10 = plt.get_cmap("tab10")
tab10_hex = np.array(rgb_array_to_hex(np.array([tab10(i) for i in range(10)])))


## Load shared objects and cached percolation results

These objects are reused across the figure: canonical CCF percolation curves for panel `a`, optimized-region score sweeps for panel `b`, section maps and stack plots for panels `c` through `h`, and pairwise comparisons for panels `i`, `j`, and `k`.

In [ ]:
tmg_path = DATA_DIR / "TMG2"
region_score_path = DATA_DIR / "OptResults" / "region_percolation_scores_reduced.csv"
canonical_gp_dir = DATA_DIR / "AllenRefSpace" / "GraphPercolation"
region_gp_dir = DATA_DIR / "OptResults" / "Opt_GPs" / "Regions"
topdown_cache_path = DATA_DIR / "ccf_topdown.npz"
bottomup_cache_path = DATA_DIR / "ccf_buttomup.npz"

TMG = TissueMultiGraph(basepath=str(tmg_path))
TMG.load_geoms()
sections = list(TMG.unqS)
example_section = next(section for section in sections if str(section).endswith(".36"))
original_cell_tax = TMG.layer_taxonomy_mapping[0]

region_scores = pd.read_csv(region_score_path)
region_scores = region_scores.loc[
    ~region_scores["algorithm"].isin(["LeidenLocalfreqPca15K20Cosine", "LeidenLocalfreqK20Cosine"])
].copy()

TMG.update_current_type(0, "opt_cell")
TMG.update_current_type(1, "opt_region")

type_mat = np.zeros((TMG.N[0], 2), dtype=int)
type_mat[:, 0] = TMG.Layers[0].Type.astype(int)
type_mat[:, 1] = TMG.Layers[1].Type[TMG.Layers[1].Upstream].astype(int)

CCF_TAXS, ccf_type_mat = build_ccf_type_matrix(TMG)

avg_scores = {}
for idx, level in enumerate(CCF_LEVELS):
    output_dir = canonical_gp_dir / level
    output_dir.mkdir(parents=True, exist_ok=True)
    df_level = TMG.Layers[0].run_percolation(
        label_vec=ccf_type_mat[:, idx],
        output_pth=str(output_dir),
        redo=False,
        compute_type_entropy=True,
    )
    df_level = df_level.set_index("section")
    avg_scores[level] = df_level["raw_score"].mean()

entropies = {
    level: entropy(np.unique(TMG.Layers[0].adata.obs[level], return_counts=True)[1], base=2)
    for level in CCF_LEVELS
}
n_types = {level: TMG.Layers[0].adata.obs[level].nunique() for level in CCF_LEVELS}

GPmat = load_gp_matrix(sections, CCF_LEVELS, canonical_gp_dir)
type_ix_td, best_score_td, type_vec_td = load_merge_cache_or_compute(topdown_cache_path, GPmat, top_down=True)
type_ix_bu, best_score_bu, type_vec_bu = load_merge_cache_or_compute(bottomup_cache_path, GPmat, top_down=False)

top_down_gp_dir = canonical_gp_dir / "top_down"
top_down_gp_dir.mkdir(parents=True, exist_ok=True)
df_top_down = TMG.Layers[0].run_percolation(
    label_vec=type_vec_td,
    output_pth=str(top_down_gp_dir),
    redo=False,
    compute_type_entropy=False,
)
df_top_down = df_top_down.set_index("section")

GPtd = load_gp_array_from_dir(sections, top_down_gp_dir, ".npz")
GPregion = load_gp_array_from_dir(sections, region_gp_dir, ".npy.npz")

MapInfoBySection = {
    "region": [gp.score() for gp in GPregion],
    "structure": [gp.score() for gp in GPmat[:, CCF_LEVELS.index("parcellation_structure")]],
    "td": df_top_down.loc[sections, "raw_score"].to_numpy(),
}
sec_counts = np.array([np.sum(TMG.Layers[0].Section == section) for section in sections])

TMG.update_current_type(0, "subclass")
type_mat_ref = np.zeros((TMG.N[0], 2), dtype=int)
type_mat_ref[:, 0] = TMG.Layers[0].Type.astype(int)
type_mat_ref[:, 1] = np.unique(TMG.Layers[0].adata.obs["parcellation_structure"], return_inverse=True)[1]

Hs_td_structure = tmgu.entropy_and_mi(
    np.vstack((type_vec_td, type_mat_ref[:, 1])).T,
    "ccf_topdown",
    "ccf_structure",
)
Hs_td_region = tmgu.entropy_and_mi(
    np.vstack((type_vec_td, type_mat[:, 1])).T,
    "ccf_topdown",
    "region",
)
Hs_region_to_regions = tmgu.entropy_and_mi(
    np.vstack((type_mat[:, 1], type_mat_ref[:, 1])).T,
    "opt_region",
    "ccf_structure",
)

TMG.update_current_type(0, original_cell_tax)

print(f"Loaded TMG from: {tmg_path}")
print(f"Sections: {len(sections)}")
print(f"Representative section: {example_section}")
print(f"Region score table: {region_scores.shape[0]} rows")
print(f"Top-down cache: {topdown_cache_path}")
print(f"Bottom-up cache: {bottomup_cache_path}")


## Panel a: map information as a function of coding entropy

This reproduces the summary scatter from the canonical CCF hierarchy plus the top-down and bottom-up refinements.

In [ ]:
x_entropy = np.array([entropies[level] for level in CCF_LEVELS])
y_scores = np.array([avg_scores[level] for level in CCF_LEVELS])

_, cnt_td = np.unique(type_vec_td, return_counts=True)
_, cnt_bu = np.unique(type_vec_bu, return_counts=True)

x_entropy = np.hstack((x_entropy, entropy(cnt_td, base=2), entropy(cnt_bu, base=2)))
y_scores = np.hstack((y_scores, best_score_td, best_score_bu))

fig, ax = plt.subplots(figsize=(2, 2))
colors = tab10_hex[[6, 3, 4, 1, 5, 2, 7]]
ax.scatter(x_entropy, y_scores, s=60, c=colors)
ax.plot(x_entropy[:5], y_scores[:5], "--", linewidth=1.0, color=tab10_hex[0])
ax.set_xlabel("Entropy", fontsize=7)
ax.set_ylabel("Map Information", fontsize=7)
ax.set_xticks([0, 3, 6, 9])
ax.set_yticks([0, 3, 6, 9])
ax.tick_params(axis="both", labelsize=6)
plt.tight_layout()
save_png(fig, PANEL_PATHS["a"])
plt.show()

PANEL_PATHS["a"]


## Panel b: optimized-region sweep

Faint points show individual clustering solutions and solid curves show smoothed trends within each algorithm family.

In [ ]:
algos = region_scores["algorithm"].unique()
colors = cc.glasbey_bw[: len(algos)]
spline_smooths = {}

for algo in algos:
    sub_df = region_scores[region_scores["algorithm"] == algo].copy()
    x = sub_df["type_distribution_entropy"].to_numpy(dtype=float)
    y = sub_df["raw_score_weighted_mean"].to_numpy(dtype=float)
    sort_idx = np.argsort(x)
    x = x[sort_idx]
    y = y[sort_idx]
    spline_smooths[algo] = smooth_curve(x, y, frac=0.2)

fig, ax = plt.subplots(figsize=(2.8, 2.0))

for color, algo in zip(colors, algos):
    sub_df = region_scores[region_scores["algorithm"] == algo]
    ax.scatter(
        sub_df["type_distribution_entropy"],
        sub_df["raw_score_weighted_mean"],
        color=color,
        alpha=0.25,
        s=10,
        linewidth=0,
    )
    x_smooth, y_smooth = spline_smooths[algo]
    ax.plot(x_smooth, y_smooth, color=color, linewidth=1.1, alpha=0.9, zorder=3)

ax.set_xlabel("Region Type Entropy", fontsize=7)
ax.set_ylabel("Map Information", fontsize=7)
ax.set_ylim([0, 9])
ax.tick_params(axis="both", which="major", labelsize=6)
ax.tick_params(axis="both", which="minor", labelsize=6)
plt.tight_layout()
save_png(fig, PANEL_PATHS["b"])
plt.show()

PANEL_PATHS["b"]


## Panels c through h: representative section maps and anterior-posterior abundance profiles

These panels compare the three region taxonomies used in Figure 4, ordered to match the attached figure:

- `c`, `d` — `ccf_structure`
- `e`, `f` — `ccf_topdown`
- `g`, `h` — `opt_region`


In [ ]:
for tax_name, map_key, stack_key in SECTION_PANEL_SPECS:
    make_section_map(TMG, example_section, tax_name, map_key)
    make_stackplot(TMG, tax_name, stack_key)

TMG.update_current_type(0, original_cell_tax)


## Panels i, j, and k: taxonomy-size distribution, entropy decomposition, and percolation curves

These panels compare `opt_region`, `ccf_structure`, and the `ccf_topdown` refinement.

In [ ]:
unq_region, cnt_region = np.unique(type_mat[:, 1], return_counts=True)
unq_structure, cnt_structure = np.unique(ccf_type_mat[:, CCF_LEVELS.index("parcellation_structure")], return_counts=True)
unq_td, cnt_td = np.unique(type_vec_td, return_counts=True)

data = [cnt_region / 1e5, cnt_structure / 1e5, cnt_td / 1e5]
labels = ["opt_region", "ccf_structure", "ccf_topdown"]
n_type_counts = [len(values) for values in data]
max_n = max(n_type_counts)
widths = [1.2 * (n / max_n) for n in n_type_counts]
positions = [0, 1, 2]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]

fig, ax = plt.subplots(figsize=(2, 2))
parts = ax.violinplot(data, positions=positions, widths=widths, showmedians=False, showextrema=False)

for part, color in zip(parts["bodies"], colors):
    part.set_facecolor(color)
    part.set_alpha(1.0)

for label in ax.get_yticklabels():
    label.set_fontsize(6)

ax.set_xticks(positions)
ax.set_xticklabels(labels, rotation=20, ha="center", fontsize=7)
for ticklabel, color in zip(ax.get_xticklabels(), colors):
    ticklabel.set_color(color)

ax.set_ylabel("Cells per type x$10^5$", fontsize=7)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for idx, (count, values) in enumerate(zip(n_type_counts, data)):
    ax.text(idx + 0.4, 2, f"N={count}", ha="center", va="bottom", fontsize=6, color=colors[idx])

plt.tight_layout()
save_png(fig, PANEL_PATHS["i"])
plt.show()

PANEL_PATHS["i"]


In [ ]:
bars = ["Opt vs Struct", "Struct vs TTD", "TD vs Struct"]
bar_colors = [
    (tab10_hex[0], tab10_hex[1]),
    (tab10_hex[1], tab10_hex[2]),
    (tab10_hex[2], tab10_hex[0]),
]

stack_vals = [
    [
        Hs_region_to_regions["H_opt_region_given_ccf_structure"],
        Hs_region_to_regions["MI"],
        Hs_region_to_regions["H_ccf_structure_given_opt_region"],
    ],
    [
        Hs_td_region["H_region_given_ccf_topdown"],
        Hs_td_region["MI"],
        Hs_td_region["H_ccf_topdown_given_region"],
    ],
    [
        Hs_td_structure["H_ccf_structure_given_ccf_topdown"],
        Hs_td_structure["MI"],
        Hs_td_structure["H_ccf_topdown_given_ccf_structure"],
    ],
]

fig, ax = plt.subplots(figsize=(2, 2))
bottoms = [0.0, 0.0, 0.0]

for idx in range(len(bars)):
    val0 = stack_vals[idx][0]
    ax.bar(bars[idx], val0, color=bar_colors[idx][0], edgecolor=bar_colors[idx][0])
    bottoms[idx] = val0

for idx in range(len(bars)):
    mi_val = stack_vals[idx][1]
    ax.bar(
        bars[idx],
        mi_val,
        bottom=bottoms[idx],
        color=bar_colors[idx][0],
        edgecolor=bar_colors[idx][1],
        hatch="///",
        alpha=1.0,
        linewidth=1,
        hatch_linewidth=2.5,
    )
    bottoms[idx] += mi_val

for idx in range(len(bars)):
    val1 = stack_vals[idx][2]
    ax.bar(bars[idx], val1, bottom=bottoms[idx], color=bar_colors[idx][1], edgecolor=bar_colors[idx][1])
    bottoms[idx] += val1

ax.set_ylabel("Entropy [bits]", fontsize=7)
ax.set_ylim(0, 11)
ax.tick_params(axis="y", labelsize=6)
ax.tick_params(axis="x", labelsize=7)
for label in ax.get_xticklabels():
    label.set_rotation(30)

plt.tight_layout()
save_png(fig, PANEL_PATHS["j"])
plt.show()

PANEL_PATHS["j"]


In [ ]:
weights = sec_counts / sec_counts.sum()
structure_idx = CCF_LEVELS.index("parcellation_structure")

opt_real = np.vstack([gp.ent_real for gp in GPregion])
opt_perm = np.vstack([gp.ent_perm for gp in GPregion])
structure_real = np.vstack([gp.ent_real for gp in GPmat[:, structure_idx]])
structure_perm = np.vstack([gp.ent_perm for gp in GPmat[:, structure_idx]])
td_real = np.vstack([gp.ent_real for gp in GPtd])
td_perm = np.vstack([gp.ent_perm for gp in GPtd])

fig, ax = plt.subplots(figsize=(2, 2))
ax.plot(GPregion[0].pbond_vec[1:-1], (opt_real * weights[:, None]).sum(axis=0)[:-1], linewidth=0.75, color=tab10_hex[0])
ax.plot(GPregion[0].pbond_vec[1:-1], (structure_real * weights[:, None]).sum(axis=0)[:-1], linewidth=0.75, color=tab10_hex[1])
ax.plot(GPregion[0].pbond_vec[1:-1], (td_real * weights[:, None]).sum(axis=0)[:-1], linewidth=0.75, color=tab10_hex[2])

ax.plot(GPregion[0].pbond_vec[1:], (opt_perm * weights[:, None]).sum(axis=0), "--", linewidth=0.75, color=tab10_hex[0])
ax.plot(GPregion[0].pbond_vec[1:], (structure_perm * weights[:, None]).sum(axis=0), "--", linewidth=0.75, color=tab10_hex[1])
ax.plot(GPregion[0].pbond_vec[1:], (td_perm * weights[:, None]).sum(axis=0), "--", linewidth=0.75, color=tab10_hex[2])

ax.set_xlabel("Proximity", fontsize=7)
ax.set_ylabel("Map Information", fontsize=7)
ax.tick_params(axis="both", labelsize=6)
plt.tight_layout()
save_png(fig, PANEL_PATHS["k"])
plt.show()

PANEL_PATHS["k"]


## Assemble the Figure 4 mock

This final cell arranges the saved panel PNGs into a composite preview with the same panel organization as the attached reference image.

In [ ]:
missing_panels = [label for label in "abcdefghijk" if not PANEL_PATHS[label].exists()]
if missing_panels:
    raise FileNotFoundError(f"Missing panel images: {missing_panels}")

fig = plt.figure(figsize=(14, 13), constrained_layout=True)
outer = fig.add_gridspec(
    nrows=4,
    ncols=3,
    height_ratios=[1.0, 1.0, 1.0, 1.0],
    width_ratios=[1.0, 1.45, 0.95],
)

ax_a = fig.add_subplot(outer[0, 0])
show_image(ax_a, PANEL_PATHS["a"])
add_panel_label(ax_a, "a")

ax_b = fig.add_subplot(outer[0, 1])
show_image(ax_b, PANEL_PATHS["b"])
add_panel_label(ax_b, "b")

ax_i = fig.add_subplot(outer[0, 2])
show_image(ax_i, PANEL_PATHS["i"])
add_panel_label(ax_i, "i")

ax_c = fig.add_subplot(outer[1, 0])
show_image(ax_c, PANEL_PATHS["c"])
add_panel_label(ax_c, "c")

ax_d = fig.add_subplot(outer[1, 1])
show_image(ax_d, PANEL_PATHS["d"])
add_panel_label(ax_d, "d")

ax_j = fig.add_subplot(outer[1, 2])
show_image(ax_j, PANEL_PATHS["j"])
add_panel_label(ax_j, "j")

ax_e = fig.add_subplot(outer[2, 0])
show_image(ax_e, PANEL_PATHS["e"])
add_panel_label(ax_e, "e")

ax_f = fig.add_subplot(outer[2, 1])
show_image(ax_f, PANEL_PATHS["f"])
add_panel_label(ax_f, "f")

ax_g = fig.add_subplot(outer[3, 0])
show_image(ax_g, PANEL_PATHS["g"])
add_panel_label(ax_g, "g")

ax_h = fig.add_subplot(outer[3, 1])
show_image(ax_h, PANEL_PATHS["h"])
add_panel_label(ax_h, "h")

ax_k = fig.add_subplot(outer[2:, 2])
show_image(ax_k, PANEL_PATHS["k"])
add_panel_label(ax_k, "k")

save_png(fig, PANEL_PATHS["mock"])
plt.show()

PANEL_PATHS["mock"]
